In [ ]:
import polars as pl
import numpy as np
import datetime

# for plotting large dataset
import altair as alt

alt.data_transformers.enable("vegafusion")

# Introduction to polars

The observations have been made by a GNSS receiver placed on the roof of a building at CNES (Toulouse), and recorded in `RINEX` format (Receiver Independent Exchange format). The RINEX file has been post-processed by `PRX` (processed RINEX, [github](https://github.com/navforge/prx)), a program developed by GNSS enthusiasts, to compute all the parameters required to compute a GNSS position. This produced a file, saved in `CSV` format (comma-separated values), thanks to the following commands:

```
uv run python src/prx/main.py --observation_file_path <path_to_rinex_file> 
```

We are going to import this file in a data structure used by the `polars` package: the `polars.DataFrame`.

In [ ]:
# load data
data_prx = pl.read_csv(
    "./data/TLSE00FRA_R_20240010000_01D_30S_MO.csv.gz",
    comment_prefix="#",
    try_parse_dates=True,
).rename({"time_of_reception_in_receiver_time": "epoch"})

# show the first 5 lines
data_prx.head()

The resulting `pl.DataFrame` contains a data table.
Each rows contains a single GNSS observation and all the associated data. You can see the list of all the available parameters as column headers.

In [ ]:
data_prx.columns


We are now going to learn how to use `polars` feature in order to
- access the data in the columns
- filter out some rows
- perform some operations
- plot the data

# Accessing data
To access a value inside a particular cell, you can use the `df.item(row,col)` notation.  
> ***Instruction:*** Try to access:
> - epoch of the first and last rows of `data_prx`

> ***Hint:***
> - The epochs are in column `epoch`.

Note: remember that Python uses a zero indexing convention.

In [ ]:
# first_epoch =
# last_epoch =

# print(f"First epoch: {first_epoch}")
# print(f"Last epoch : {last_epoch}")

# assert first_epoch == datetime.datetime.fromisoformat ("2024-01-01 00:00:00")
# assert last_epoch == datetime.datetime.fromisoformat ("2024-01-01 23:59:30")

To access a row, you can use the `df[index_slice]` notation.  
> ***Instruction:*** Try to access:
> - the first line of `data_prx`
> - the first two lines of `data_prx`

In [ ]:
# first_line =
# assert first_line.item(0, "L_obs_cycles") == 207686520.782

# first_two_lines =
# assert (
#     first_two_lines.item(0, "L_obs_cycles"),
#     first_two_lines.item(1, "L_obs_cycles"),
# ) == (207686520.782, 168762309.784)

To access a particular column, you can call it using either
- `.select()`
- or `.get_column()`

`.get_column()` returns another `polars` structure, called the `polars.Series`.
A `polars.Series` is just a `DataFrame` with a single column.

`.select()` returns a `polars.DataFrame`, even if it has a single column.

> ***Instruction:***
> - Using `.head(5)`, display the 5 first values of the `prn` column of `data_prx`.

In [ ]:
# prn_series =

You can also access several columns by putting a `list` of column names in `.select()`.
This will return another `DataFrame` with several columns.

> ***Instruction:*** 
> - Using `.head(5)`, display the 5 first values of the `prn`, `constellation` and `rnx_obs_identifier` columns of data_prx

Note: the RINEX observation identifiers can be found in section 5.1 (p22) of the document `resources/rinex_3.04.pdf`.

In [ ]:
# prn_constellation_obsid =

# Conditional filtering
`polars` allow to quickly filter the rows thanks to the `.filter(condition)` method, where `condition` is a `Series` of boolean.

For example, you can keep only the observations with high C/No thanks to the following line:  
`data_high_snr = data_prx.filter( data_prx.get_column("S_obs_dBHz") > 45 )`

A faster way to refer to the `DataFrame` column is:  
`data_high_snr = data_prx.filter( pl.col("S_obs_dBHz") > 45 )`

> ***Instruction:***
> - Try those commands and display the number of observations before and after filtering.

In [ ]:
# data_high_snr =

# print(f"There are {len(data_prx)} observations in the original dataset.")
# print(f"There are {len(data_high_snr)} observations with a C/No above 45 dbHz.")

You can also check if the value in one column is within a **list** of values with the method `Series.is_in(list)`.

> ***Instruction:***  
> - Display the number of observations from the constellations GPS (`constellation=="G"`) and Galileo (`constellation=="E"`)

In [ ]:
# data_gps =
# data_gal =
# data_gps_gal =

# print(f"There are {len(data_gps)} GPS observations and {len(data_gal)} GAL observations.")
# print(f"This sums up to {len(data_gps_gal)} GPS and GAL observations.")

You can use several conditions when filtering with `.filter(condition_1 & condition_2)`

> ***Instruction:***
> - Extract the observations containing only GPS and L1 C/A observations (corresponding `rnx_obs_identifier` is `"1C"`)

In [ ]:
# data_gps_c1c =

# print(f"There are {len(data_prx)} observations in the original dataset.")
# print(f"There are {len(data_gps_c1c)} GPS L1 C/A observations.")

# --------------ANSWERS------------------

data_gps_c1c = data_prx.filter(
    (pl.col("constellation") == "G") & (pl.col("rnx_obs_identifier") == "1C")
)

print(f"There are {len(data_prx)} observations in the original dataset.")
print(f"There are {len(data_gps_c1c)} GPS L1 C/A observations.")

# Operation on `DataFrame`
You can apply some usual operations on whole `Series`, such as `mean(), sum(), min(), max(), median(), quantile([q1,q2,…])`, etc.

> ***Instruction:***
> - Compute the minimum and maximum values of the code and doppler observation on `data_gps_c1c`.

> ***Hint:***
> - The code observations are in column `C_obs_m` and the Doppler observations in column `D_obs_hz`

In [ ]:
# code_min =
# code_max =
# doppler_min =
# doppler_max =

# print(f"The GPS L1 C/A code pseudorange observations are within {code_min:,} and {code_max:,} meters.")
# print(f"The GPS L1 C/A doppler offset observations are within {doppler_min:,} and {doppler_max:,} Hz.")

# Creating sub-groups in a `DataFrame`

The method `.groupby(expression)` creates a `polars.dataframe.group_by.GroupBy` object, which is basically the initial dataset split in different groups based on the different values given by `expression`. You can then perform operations on the different groups.

The resulting object is an iterable of 2-`tuples` where:
- the first element is a `tuple` containing the unique value of `expression`,
- the second element is a `DataFrame` corresponding to the rows of the group.

> ***Instructions:***
> - Look at the outputs of each of those lines and understand each line.

In [ ]:
# `polars.dataframe.group_by.GroupBy` object
group_by_output = data_gps.group_by("prn")

# convert to list
group_by_list = list(group_by_output)

# look at first group created by `.group_by("prn")`
first_groupby_tuple = group_by_list[0]
first_prn = first_groupby_tuple[0]
first_group = first_groupby_tuple[1]

print(
    f"The first element of the tuple is the key used by `.group_by('prn'): {first_prn}"
)
print(
    "The second element of the tuple is the DataFrame f the corresponding to the group_by key:"
)
print(first_group.select(["prn", "epoch", "constellation", "C_obs_m"]).head(5))

### Iterating on subgroups created by `.group_by()`

You can iterate on a `GroupBy` object by using a `for` loop with the following call:  
> `for key, group in df.group_by("column"):`

This will iterate over all groups created by `.group_by("column")`, with  
- `key` being the value of `"column"` of the current group
- `group` being a `DataFrame` of the current group

> ***Instructions:***  
> Use a `for` loop on `.group_by()`
> - to compute the number of observations at each epoch
> - `print()` the result in the console
>
> ***Hint:***
> - you can use `len(df)` to know the number of rows of a `DataFrame`

In [ ]:
# index=0
# for epoch, group in TO_BE_REPLACED:
#     print(f"There are {TO_BE_REPLACED} observations at epoch {epoch[0]}")
#     # stop after 5 iterations
#     index+=1
#     if index >= 5:
#         break

### Aggreation on subgroups created by `.group_by()`

You can also directly apply some processing to each group. Some usual processings are called `aggregations`, and typically return a single value for each group. They are called thanks to `df.group_by(expr).agg(fun)`.

> ***Instructions:***  
> - Create a `DataFrame` with the number of observations for each epoch.

> ***Hint:***
> - You can use `.group_by("epoch", maintain_order=True)`. The argument `maintain_order=True` will keep the data in the original order of the `DataFrame`.
> - The size of a group can be obtained using the `pl.len()` function in an `.agg()` call.
> - Try to do this plot with single line!!

In [ ]:
# n_sats =

# print(n_sats.head(5))

# assert (
#     n_sats.filter(pl.col("epoch") == datetime.datetime(2024, 1, 1))
#     .get_column(n_sats.columns[1])
#     .item()
#     == 8
# ), "At first epoch,

# Plot
`polars` has integrated plotting capabilities. You can easily plot a `DataFrame` by using the `.plot.line()` or `.plot.point()` methods. 
 
You have to specify the columns to be used as x- or y- axis: `pl.DataFrame.plot.line(x="axis_1", y="axis_2")`

> ***Instruction:***  
> - Plot the code observation vs time.

In [ ]:
# Plot the code observations vs time

⚠️ We see that since the data from several satellites are all mixed up, we do not distinguish a clear pattern!

To give a different color to each satellite, you can add `color="prn"` to `.plot.point()`.

> ***Instruction:***
> - Plot the GPS L1 C/A code observations vs time, with a different color for prn 2, 3 and 8.
>
> ***Hint:***
> - Filter `data_gps_c1c` by a specific prn value before calling `.plot()` 

In [ ]:
# Plot the code observations of prn 2,3,8 vs time


# Applying expressions to `DataFrame`

When dealing with tabular data, it is quite usual to apply a function to several rows or columns of the table. This is performed by defining `Expressions`.

## Adding a new column based computed over several other columns

Let's use an `Expression` to compute the orbital radius of the satellite for each row of the `DataFrame`.
- A column can be added by using `.with_columns(expr)`.
- The name of the new column is specified by using `.alias("name")` after the expression definition.
- The orbital radius can be obtained by taking the sum of the squared coordinates the satellite ECEF position. This simple computation can be performed using some of the prededined expressions ([see doc](https://docs.pola.rs/api/python/stable/reference/expressions/index.html)) in `polars`.

So in the end, we'll have to use:
```
data_gps_c1c.with_columns(
    (
        pl.col("sat_pos_x_m") ** 2 +
        pl.col("sat_pos_y_m") ** 2 +
        pl.col("sat_pos_z_m") ** 2
    )
    .sqrt()
    .alias("orbital_radius_m")
)
```

> ***Instruction:***
> - Compute the orbital radius of each GPS satellite.
> - Add it as an additional column named `orbital_radius_m` to `data_gps_c1c`.
> - Plot the orbital radius vs time for each GPS satellite.

In [ ]:
# Plot the orbital radius vs time for each GPS satellite

# Optional challenge 🤯
(This question is optional. You can skip it to save time.)

This challenge requires to reuse the various `polars` methods presented in this tutorial. It illustrates how several operations can be chained together to build powerful data-processing pipelines, often with remarkably concise code.

> ***Instructions:***
> - Compute the mean (accross epochs) orbital radius for each satellite of each constellation
> - Plot the mean orbital radius for each satellite as a bar chart, with a different color for each constellation.
> - Can you explain the differences between the satellites/constellations ?

> Hint:
> - You'll have to process `data_prx`, which contains the data for all constellations.
> - The "`polars`" way to do it is to combine `.group_by()` and `.agg()`.
> - Use `.group_by()` on several columns. You want to compute a value for each satellite of each constellation, so the groups are defined by `["constellation","prn"]`
> - In addition to the orbital radius computation, you want to compute the mean for each group. You'll have to add this extra step in the `Expression` defined in `.agg(expr)`.
> - You can sort the final dataframe by constellation and prn for a nicer plot.

👏 Congrats! You've reached the end of this notebook 🎉

Now that you know the basics of `polars`, you can follow the `instructions_polars/chapter_0.md` instructions.